# Complexity-05 — Compter est plus dur que vérifier : #P et la permanente

**Navigation** : [<< Complexity-04b — Secrétaire matroïdal et k-server](Complexity-04b-OnlineConjectures-Secretary-KServer-Python.ipynb) | [Index de la série](README.md) | [Version Python](Complexity-05-CountingHarder-Permanent-Python.ipynb) | [Complexity-05b — La permanente, frontière quantique >>](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb)

> **Public : Licence.** La formule de Ryser, le boson sampling et la frontière quantique sont
> conservés pour l'approfondissement [05b](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb) ;
> ce notebook s'arrête à la frontière #P.

## Ce que ce notebook suppose

- [02 — Vérifier ou trouver](Complexity-02-P-NP-Reduction-Python.ipynb) : le **certificat** d'une
  instance, ce que signifie **vérifier** en temps polynomial, les classes **P** et **NP** ;
- lire un programme C# simple (boucles, fonctions, tableaux).

## Ce que vous saurez faire à la fin

1. définir la classe **#P** — **compter** les témoins d'une instance, et non plus seulement
   décider s'il en existe ;
2. écrire le **déterminant** et la **permanente** : la même somme de $n!$ produits, à un
   signe près ;
3. **mesurer** sur un banc réel l'écart entre un calcul polynomial et un calcul factoriel ;
4. énoncer le théorème de Valiant (**cité**) : la permanente est #P-difficile, même en 0/1 ;
5. relier un comptage naturel — les **couplages parfaits** d'un graphe biparti — à une
   permanente.

**Environnement** : .NET 9+, noyau C# et MathNet.Numerics 5.0.0.

**Durée estimée** : 45 minutes.

In [1]:
#r "nuget:MathNet.Numerics,5.0.0"
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.Linq;
using System.Text;
using System.Security;
using MathNet.Numerics.LinearAlgebra;
using MathNet.Numerics.LinearAlgebra.Double;
using Microsoft.DotNet.Interactive.Formatting;
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
Console.WriteLine($".NET {Environment.Version}; MathNet.Numerics {typeof(DenseMatrix).Assembly.GetName().Version}; fournisseur {MathNet.Numerics.Providers.LinearAlgebra.LinearAlgebraControl.Provider}");

Installed Packages MathNet.Numerics, 5.0.0

.NET 10.0.14; MathNet.Numerics 5.0.0.0; fournisseur Managed



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'MathNet.Numerics' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy



## 1. De vérifier à compter : la classe #P

Le notebook 02 séparait deux gestes : **trouver** une solution (aucun algorithme polynomial
connu sur les problèmes NP-complets) et **vérifier** un certificat proposé (facile,
polynomial). Un certificat que le vérificateur accepte s'appelle ici un **témoin**. Décider
s'il existe un témoin, c'est un problème de **NP**. Mais le certificat soulève une question
que NP ne pose même pas : **combien** y en a-t-il ?

Compter les témoins définit une nouvelle classe de fonctions :

> **#P** (lire *sharp-P*) : les fonctions qui à une instance $x$ associent le **nombre de
> témoins** de $x$.

Le comptage est au moins aussi difficile que la décision — un compte nul répond *non*, un
compte strictement positif répond *oui* — mais il est plus fin : il distingue des instances
que la décision confond. Déroulons un exemple complet, à la main, avant toute généralité :
l'expression à trois variables

$$\varphi(a, b, c) \;=\; (a \vee b) \wedge (\neg a \vee c)$$

Un certificat est un triplet de booléens ; un témoin, un triplet accepté par les deux
clauses. Il n'y en a que huit : énumérons-les tous.

In [2]:
int temoins=0;
Console.WriteLine(" a  b  c | a ou b | non a ou c | témoin");
for(int a=0;a<2;a++)for(int b=0;b<2;b++)for(int c=0;c<2;c++){
 int c1=(a==1||b==1)?1:0,c2=(a==0||c==1)?1:0;int t=c1*c2;temoins+=t;
 Console.WriteLine($" {a}  {b}  {c} | {c1,6} | {c2,10} | {t,6}");}
if(temoins!=4)throw new Exception("Compte de témoins incorrect");
Console.WriteLine($"Décision : {(temoins>0 ? "OUI" : "NON")}; comptage : {temoins} témoins sur 8 certificats.");

 a  b  c | a ou b | non a ou c | témoin


 0  0  0 |      0 |          1 |      0


 0  0  1 |      0 |          1 |      0


 0  1  0 |      1 |          1 |      1


 0  1  1 |      1 |          1 |      1


 1  0  0 |      1 |          0 |      0


 1  0  1 |      1 |          1 |      1


 1  1  0 |      1 |          0 |      0


 1  1  1 |      1 |          1 |      1


Décision : OUI; comptage : 4 témoins sur 8 certificats.


**Lecture du résultat.** Quatre témoins sur huit certificats — et le tableau qui les exhibe
répond aux deux questions d'un coup : l'instance est satisfiable (au moins un témoin), et son
**compte** de satisfaisabilité vaut quatre. Mais l'énumération a payé $2^n$ lectures : le mur
exponentiel du notebook 02. La vraie question de #P est là : peut-on obtenir le **nombre**
sans tout énumérer ? Cette question vit sur un objet inattendu — une simple somme de produits
de matrice.

## 2. Le déterminant : des signes qui arrangent

Le déterminant d'une matrice $n \times n$ est, lui aussi, défini comme une somme sur les
permutations $\sigma$ de $\{0, \dots, n-1\}$ :

$$\det A \;=\; \sum_{\sigma} \mathrm{signe}(\sigma) \prod_{i=0}^{n-1} A[i,\sigma(i)]$$

Chaque permutation apporte son produit des $n$ entrées, multiplié par son **signe** ($+1$ ou
$-1$ selon la parité des échanges qui la composent). En $2 \times 2$, les deux permutations
redonnent le produit « en croix » connu :

$$\det\begin{pmatrix} a & b \\ c & d \end{pmatrix} = ad - bc$$

La définition parcourt $n!$ termes — et pourtant le déterminant se calcule en
**polynomial** : l'élimination de Gauss factorise la matrice en triangles en environ
$\tfrac{2}{3}n^3$ opérations, et c'est elle que MathNet.Numerics exécute. Vérifions que le polynomial
se **voit** sur la machine : les tailles doublent, un cube promet un facteur $2^3 = 8$ —
voyons ce que la machine rend. (Le banc chronomètre la factorisation LU et le log-déterminant, le **log**-déterminant :
la forme stable du même calcul de Gauss — le déterminant lui-même déborderait des flottants
à ces tailles.)

In [3]:
(int signe,double logAbs) SLog(Matrix<double> a){
 var lu=a.LU(); var d=lu.U.Diagonal(); if(d.Any(x=>x==0))return(0,double.NegativeInfinity);
 int cycles=0;var vu=new bool[a.RowCount];for(int i=0;i<a.RowCount;i++){if(vu[i])continue;cycles++;for(int j=i;!vu[j];j=lu.P[j])vu[j]=true;}
 int signe=(a.RowCount-cycles)%2==0?1:-1;foreach(double v in d)signe*=Math.Sign(v);
 return(signe,d.Sum(v=>Math.Log(Math.Abs(v))));}
foreach(var (a,attendu) in new[]{(new double[,]{{1,2},{3,4}},-2.0),(new double[,]{{0,1},{1,0}},-1.0),(new double[,]{{1,0},{0,1}},1.0),(new double[,]{{1,2},{2,4}},0.0)}){
 var r=SLog(DenseMatrix.OfArray(a));if(r.signe!=Math.Sign(attendu)||(attendu!=0&&Math.Abs(r.logAbs-Math.Log(Math.Abs(attendu)))>1e-12))throw new Exception("Fixture LU incorrecte");
 Console.WriteLine($"Fixture det={attendu}: signe={r.signe}, log|det|={r.logAbs}");}
var rng=new Random(5); Matrix<double> Alea(int n)=>DenseMatrix.Create(n,n,(i,j)=>rng.NextDouble());
SLog(Alea(100)); double precedent=0;
Console.WriteLine($"{ "n",6} | {"minimum (s)",12} | {"x doublement",13} | {"temps/n³",12} | signe | log|det|");
foreach(int n in new[]{100,200,400,800}){var a=Alea(n);var temps=new List<double>();(int signe,double logAbs) resultat=(0,0);
 for(int essai=0;essai<10;essai++){var sw=Stopwatch.StartNew();resultat=SLog(a);sw.Stop();temps.Add(sw.Elapsed.TotalSeconds);}
 double d=temps.Min();if(resultat.signe==0||!double.IsFinite(resultat.logAbs))throw new Exception("LU singulière ou non finie");
 Console.WriteLine($"{n,6} | {d,12:F6} | {(precedent==0?"-":(d/precedent).ToString("F2")),13} | {d/(n*(double)n*n),12:E3} | {resultat.signe,5} | {resultat.logAbs:F4}");precedent=d;}
Console.WriteLine("LU MathNet : 10 essais par taille après échauffement; minimum observé, sans garantie de monotonie.");

Fixture det=-2: signe=-1, log|det|=0.6931471805599455


Fixture det=-1: signe=-1, log|det|=0


Fixture det=1: signe=1, log|det|=0


Fixture det=0: signe=0, log|det|=-Infinity


     n |  minimum (s) |  x doublement |     temps/n³ | signe | log|det|


   100 |     0.000420 |             - |   4.201E-010 |     1 | 56.9407


   200 |     0.003001 |          7.14 |   3.751E-010 |    -1 | 186.8170


   400 |     0.021341 |          7.11 |   3.335E-010 |    -1 | 502.9487


   800 |     0.281572 |         13.19 |   5.499E-010 |     1 | 1279.9710


LU MathNet : 10 essais par taille après échauffement; minimum observé, sans garantie de monotonie.


**Lecture du résultat.** Le tableau donne le minimum de dix mesures et le rapport à chaque doublement. La décomposition LU est cubique en théorie ; les rapports observés dépendent du matériel et ne constituent pas une preuve asymptotique. Le logarithme est obtenu par la somme des logarithmes des valeurs absolues de la diagonale U, avec le signe de la permutation et des pivots ; aucun déterminant potentiellement débordant n’est calculé. NumPy et MathNet utilisent des moteurs différents : leurs temps ne sont pas directement interchangeables.

## 3. La permanente : la même somme, sans les signes

Retirons les signes, gardons tout le reste. La **permanente** additionne les mêmes $n!$
produits, sans signe de permutation :

$$\mathrm{perm}(A) \;=\; \sum_{\sigma} \prod_{i=0}^{n-1} A[i,\sigma(i)]$$

En $2 \times 2$ : $\mathrm{perm}\begin{pmatrix} a & b \\ c & d \end{pmatrix} = ad + bc$ —
contre $ad - bc$ pour le déterminant. Déroulons une $3 \times 3$ à la main, permutation par
permutation :

$$A = \begin{pmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \\ 7 & 8 & 9 \end{pmatrix}$$

| $\sigma$ | produit | valeur |
|---|---|---|
| $(0,1,2)$ | $1 \cdot 5 \cdot 9$ | $45$ |
| $(0,2,1)$ | $1 \cdot 6 \cdot 8$ | $48$ |
| $(1,0,2)$ | $2 \cdot 4 \cdot 9$ | $72$ |
| $(1,2,0)$ | $2 \cdot 6 \cdot 7$ | $84$ |
| $(2,0,1)$ | $3 \cdot 4 \cdot 8$ | $96$ |
| $(2,1,0)$ | $3 \cdot 5 \cdot 7$ | $105$ |

La somme vaut $450$. La même matrice a un déterminant **nul** — les signes s'y annihilent
totalement, la permanente additionne tout. Le banc suivant exécute la définition **telle
quelle**, un terme par permutation : copie pédagogique déclarée, c'est précisément le point
du cours (l'approfondissement 05b réutilise cette définition, puis la contourne par la
formule de Ryser — hors périmètre ici).

In [4]:
long Factorielle(int n){long x=1;for(int i=2;i<=n;i++)x*=i;return x;}
long Permanente(int[,] a){int n=a.GetLength(0);var p=Enumerable.Range(0,n).ToArray();long somme=0;
 void Parcours(int k){if(k==n){long prod=1;for(int i=0;i<n;i++)prod*=a[i,p[i]];somme+=prod;return;}for(int j=k;j<n;j++){(p[k],p[j])=(p[j],p[k]);Parcours(k+1);(p[k],p[j])=(p[j],p[k]);}}
 Parcours(0);return somme;}
var id3=new int[,]{{1,0,0},{0,1,0},{0,0,1}};var m22=new int[,]{{1,2},{3,4}};var m33=new int[,]{{1,2,3},{4,5,6},{7,8,9}};
foreach(var (nom,a,attendu) in new[]{("identité 3×3",id3,1L),("2×2",m22,10L),("3×3",m33,450L)}){long x=Permanente(a);if(x!=attendu)throw new Exception("Permanente incorrecte");Console.WriteLine($"{nom}: permanente={x} (attendu {attendu})");}
Console.WriteLine($"Déterminant de 1..9 : {SLog(DenseMatrix.OfArray(new double[,]{{1,2,3},{4,5,6},{7,8,9}})).signe} (matrice singulière)");
for(int n=1;n<=5;n++){var a=new int[n,n];for(int i=0;i<n;i++)for(int j=0;j<n;j++)a[i,j]=1;long x=Permanente(a);if(x!=Factorielle(n))throw new Exception("Tout-un");Console.WriteLine($"tout-un n={n}: {x} = {n}!");}

identité 3×3: permanente=1 (attendu 1)


2×2: permanente=10 (attendu 10)


3×3: permanente=450 (attendu 450)


Déterminant de 1..9 : 0 (matrice singulière)


tout-un n=1: 1 = 1!


tout-un n=2: 2 = 2!


tout-un n=3: 6 = 3!


tout-un n=4: 24 = 4!


tout-un n=5: 120 = 5!


**Lecture du résultat.** Les comptes canoniques tombent les uns après les autres : l'identité
ne laisse qu'un seul terme non nul (celui de la permutation identité), les matrices tout-un
valent $n!$ (autant de produits que de permutations, tous égaux à 1), et le contraste final
est net — permanente $450$, déterminant $0$ sur la même matrice. La définition est correcte ;
reste à en payer le prix, et à le comparer à celui de Gauss.

### Exercice 1 — Compter à la main

Affectez à `permDef` une fonction qui parcourt les permutations, multiplie les entrées correspondantes et somme les produits. Les fixtures vérifient plusieurs matrices distinctes.

In [5]:
Func<int[,],long?> permDef = null; // TODO étudiant : énumérer les permutations et sommer les produits.
if(permDef is null)Console.WriteLine("Exercice à compléter : écrire permDef.");
else {foreach(var (a,attendu) in new[]{(id3,1L),(new int[,]{{1,1,1},{1,1,1},{1,1,1}},6L),(new int[,]{{1,1,1},{1,1,0},{1,0,1}},3L),(m22,10L),(m33,450L)}){var obtenu=permDef(a);if(obtenu!=attendu)throw new Exception($"Permanente incorrecte : {obtenu}, attendu {attendu}");}Console.WriteLine("Cinq matrices distinctes vérifiées, dont 1, 6, 3, 10 et 450.");}

Exercice à compléter : écrire permDef.


## 4. L'écart mesuré : polynomial contre factoriel

Mêmes tailles, mêmes matrices, deux algorithmes : d'un côté la factorisation LU de MathNet (élimination de
Gauss, environ $\tfrac{2}{3}n^3$ opérations), de l'autre `permanente` (la définition, $n!$
termes). Les tailles vont de $n = 2$ à $n = 10$ — au-delà, la définition sortirait du budget
d'un notebook, et l'extrapolation prendra le relais. Chronomètre au `Stopwatch`, puis
échelle logarithmique : c'est elle qui rend une factorielle lisible.

In [6]:
record SvgPermanent(string Content);
Formatter.Register<SvgPermanent>((v,w)=>w.Write(v.Content),"text/html");
string F(double x)=>x.ToString("0.###",CultureInfo.InvariantCulture);
string E(string x)=>SecurityElement.Escape(x) ?? "";
string Trace(double[] perm,double[] det){int w=760,h=420;double lo=Math.Floor(Math.Log10(perm.Concat(det).Min())),hi=Math.Ceiling(Math.Log10(perm.Concat(det).Max()));if(hi<=lo)hi=lo+1;
 double X(int i)=>85+i*590.0/(perm.Length-1);double Y(double v)=>340-(Math.Log10(v)-lo)*280/(hi-lo);
 var s=new StringBuilder($"<svg xmlns='http://www.w3.org/2000/svg' width='{w}' height='{h}' viewBox='0 0 {w} {h}'><rect width='100%' height='100%' fill='white'/><text x='380' y='25' text-anchor='middle'>Temps : permanente et LU</text><path d='M85 60 V340 H675' stroke='black' fill='none'/>");
 for(int i=0;i<perm.Length;i++)s.Append($"<text x='{F(X(i))}' y='360' text-anchor='middle'>{i+2}</text>");
 for(int k=(int)lo;k<=hi;k++){double y=340-(k-lo)*280/(hi-lo);s.Append($"<path d='M85 {F(y)} H675' stroke='#ddd'/><text x='78' y='{F(y+4)}' text-anchor='end'>10^{k}</text>");}
 foreach(var (v,col) in new[]{(perm,"#a00"),(det,"#0645ad")}){s.Append($"<polyline fill='none' stroke='{col}' stroke-width='2' points='{string.Join(" ",v.Select((x,i)=>F(X(i))+","+F(Y(x))))}'/>");foreach(var (x,i) in v.Select((x,i)=>(x,i)))s.Append($"<circle cx='{F(X(i))}' cy='{F(Y(x))}' r='3' fill='{col}'/>");}
 s.Append("<text x='380' y='395' text-anchor='middle'>taille n</text><text x='15' y='205' transform='rotate(-90 15 205)' text-anchor='middle'>temps (s), échelle logarithmique</text><text x='95' y='45' fill='#a00'>Permanente</text><text x='220' y='45' fill='#0645ad'>LU MathNet</text></svg>");return s.ToString();}
var tailles=Enumerable.Range(2,9).ToArray();var chronoPerm=new List<double>();var chronoDet=new List<double>();
Console.WriteLine($"{ "n",3} | {"n!",8} | {"permanente (s)",16} | {"x pas précédent",15} | {"LU (s)",12} | {"perm/LU",10}");
foreach(int n in tailles){var a=new int[n,n];for(int i=0;i<n;i++)for(int j=0;j<n;j++)a[i,j]=rng.Next(10);var sw=Stopwatch.StartNew();long val=Permanente(a);sw.Stop();chronoPerm.Add(sw.Elapsed.TotalSeconds);
 var d=DenseMatrix.Create(n,n,(i,j)=>a[i,j]);sw.Restart();var lu=SLog(d);sw.Stop();chronoDet.Add(sw.Elapsed.TotalSeconds);
 Console.WriteLine($"{n,3} | {Factorielle(n),8} | {chronoPerm.Last(),16:F6} | {(n==2?"-":(chronoPerm[^1]/chronoPerm[^2]).ToString("F2")),15} | {chronoDet.Last(),12:F7} | {chronoPerm.Last()/chronoDet.Last(),10:F1}");}
double cout=chronoPerm.Last()/Factorielle(10);foreach(int n in new[]{11,12,15,20})Console.WriteLine($"Extrapolation n={n}, {Factorielle(n):E2} termes : {cout*Factorielle(n):E3} secondes sur la base du coût mesuré à n=10.");
display(new SvgPermanent(Trace(chronoPerm.ToArray(),chronoDet.ToArray())));

  n |       n! |   permanente (s) | x pas précédent |       LU (s) |    perm/LU


  2 |        2 |         0.000006 |               - |    0.0000403 |        0.2


  3 |        6 |         0.000006 |            0.90 |    0.0000175 |        0.3


  4 |       24 |         0.000006 |            1.12 |    0.0000134 |        0.5


  5 |      120 |         0.000011 |            1.79 |    0.0000135 |        0.8


  6 |      720 |         0.000060 |            5.31 |    0.0000191 |        3.1


... (5 outputs left out)

Extrapolation n=12, 4.79E+008 termes : 3.542E+001 secondes sur la base du coût mesuré à n=10.


Extrapolation n=15, 1.31E+012 termes : 9.669E+004 secondes sur la base du coût mesuré à n=10.


Extrapolation n=20, 2.43E+018 termes : 1.799E+011 secondes sur la base du coût mesuré à n=10.


Temps : permanente et LU 2 3 4 5 6 7 8 9 10 10^-6 10^-5 10^-4 10^-3 10^-2 10^-1 10^0 taille n temps (s), échelle logarithmique Permanente LU MathNet

**Lecture du résultat.** Le nombre de termes passe de $(n-1)!$ à $n!$, soit un facteur $n$ par taille supplémentaire. Les temps et leurs rapports sont mesurés dans la table : la charge et les coûts fixes les font varier. Les extrapolations prolongent uniquement le coût par terme mesuré à $n=10$ ; elles ne sont pas une borne sur tous les algorithmes de permanente.

### Exercice 2 — L’échelle mesurée

Complétez `permDef` puis exécutez cette cellule : le harnais appelle réellement votre fonction sur les tailles 2 à 9, vérifie chaque réponse et mesure chaque durée avec `Stopwatch`. Les facteurs affichés montrent les mesures obtenues.

In [7]:
// TODO étudiant : fournir une fonction permDef complète à l'exercice 1.
if(permDef is null)Console.WriteLine("Exercice à compléter : implémenter permDef avant de mesurer.");
else {var mesures=new List<double>();for(int n=2;n<=9;n++){var a=new int[n,n];for(int i=0;i<n;i++)for(int j=0;j<n;j++)a[i,j]=rng.Next(1,4);long attendu=Permanente(a);var sw=Stopwatch.StartNew();long? obtenu=permDef(a);sw.Stop();if(obtenu!=attendu)throw new Exception($"Résultat erroné pour n={n}");double t=sw.Elapsed.TotalSeconds;if(!double.IsFinite(t)||t<=0)throw new Exception("Chrono non positif");mesures.Add(t);Console.WriteLine($"n={n}, résultat={obtenu}, durée mesurée={t:E4} s".Replace("{n}",n.ToString()));}Console.WriteLine("Facteurs successifs : "+string.Join(", ",Enumerable.Range(1,mesures.Count-1).Select(i=>(mesures[i]/mesures[i-1]).ToString("F2"))));}

Exercice à compléter : implémenter permDef avant de mesurer.


## 5. Valiant 1979 : compter est #P-difficile (cité)

Un banc ne mesure qu'un programme — il ne dit rien des programmes qu'on n'a pas écrits. La
question sérieuse : un « Gauss de la permanente » existe-t-il ? Une astuce qui, exploitant la
structure de la somme, la ferait tomber à un polynomial ? Le théorème de Valiant ferme cette
porte de la façon la plus spectaculaire possible :

> **Théorème (Valiant, 1979 — cité, non démontré ici).** Calculer la permanente d'une matrice
> à coefficients $0$ et $1$ est **#P-difficile** : tout problème de #P — tout comptage de
> témoins — se réduit à ce calcul en temps polynomial.

La preuve (une réduction depuis le comptage des satisfaisations d'une formule 3-SAT) est un
chef-d'œuvre technique, hors de portée d'un notebook de licence ; l'énoncé seul suffit pour
la leçon :

1. **un algorithme polynomial pour la permanente 0/1 résoudrait tout comptage** : chaque
   problème de #P s'y réduirait, et la classe entière des comptages s'effondrerait sur les
   calculs polynomiaux (FP = #P) ;
2. **les coefficients ne sont pas le problème** : des $0$ et des $1$ suffisent à la
   difficulté ;
3. le retournement est complet pour les graphes : décider s'il existe un couplage parfait
   dans un graphe biparti est **facile** (polynomial) — mais les compter, c'est
   carrément #P-difficile, et la section suivante montre pourquoi : ce comptage est une
   permanente.

La somme la plus simple du monde — des produits, sans signe de permutation — est donc
conjecturée hors de portée de tout algorithme polynomial.

## 6. Compter des couplages parfaits : c'est une permanente

Un graphe **biparti** : des sommets à gauche $u_0, \dots, u_3$, des sommets à droite
$d_0, \dots, d_3$, des arêtes entre les deux côtés seulement. Un **couplage parfait** apparie
chaque sommet de gauche à un voisin de droite, tous distincts — quatre arêtes, aucun sommet
utilisé deux fois. Les compter, c'est le comptage le plus naturel du monde : affecter des
tâches à des machines, des étudiants à des projets.

Voici le nôtre, dessiné comme un tableau à deux dimensions — une ligne par $u_i$, une colonne
par $d_j$, un point par arête :

```
        d0    d1    d2    d3
u0      ●     ●     ·     ·
u1      ●     ●     ●     ·
u2      ·     ●     ●     ●
u3      ·     ·     ●     ●
```

Ce tableau **est** la matrice d'adjacence $M$ du graphe (1 si l'arête existe, 0 sinon). Et
alors :

$$\#\{\text{couplages parfaits}\} \;=\; \mathrm{perm}(M)$$

La raison tient en deux lignes : un couplage parfait choisit pour chaque $u_i$ une
destination, toutes distinctes — autrement dit une permutation $\sigma$ ; et le produit des
$M[i,\sigma(i)]$ vaut $1$ exactement quand toutes les arêtes choisies existent, $0$ sinon.
Chaque couplage parfait contribue $1$ à la somme, chaque autre permutation $0$. Le banc
suivant le vérifie avec **deux machines indépendantes** : la permanente d'un côté, et de
l'autre une énumération brute qui construit chaque candidat sommet par sommet — chaque $u_i$
essaie chacun de ses voisins — et ne garde que les choix tous distincts.

In [8]:
var aretes=new HashSet<(int,int)>{(0,0),(0,1),(1,0),(1,1),(1,2),(2,1),(2,2),(2,3),(3,2),(3,3)};
var matriceGraphe=new int[4,4];for(int i=0;i<4;i++)for(int j=0;j<4;j++)matriceGraphe[i,j]=aretes.Contains((i,j))?1:0;
Console.WriteLine("Matrice d'adjacence :");for(int i=0;i<4;i++)Console.WriteLine(string.Join(" ",Enumerable.Range(0,4).Select(j=>matriceGraphe[i,j])));
var couplages=new List<int[]>();void Cherche(int i,int[] choix){if(i==4){if(choix.Distinct().Count()==4)couplages.Add((int[])choix.Clone());return;}for(int j=0;j<4;j++)if(aretes.Contains((i,j))){choix[i]=j;Cherche(i+1,choix);}}
Cherche(0,new int[4]);foreach(var c in couplages)Console.WriteLine(string.Join(" ",c.Select((j,i)=>$"u{i}->d{j}")));
long compte=Permanente(matriceGraphe);if(compte!=couplages.Count)throw new Exception("Désaccord couplages");Console.WriteLine($"Permanente={compte}; énumération indépendante={couplages.Count} couplages.");

Matrice d'adjacence :


1 1 0 0


1 1 1 0


0 1 1 1


0 0 1 1


u0->d0 u1->d1 u2->d2 u3->d3


u0->d0 u1->d1 u2->d3 u3->d2


u0->d0 u1->d2 u2->d1 u3->d3


u0->d1 u1->d0 u2->d2 u3->d3


u0->d1 u1->d0 u2->d3 u3->d2


Permanente=5; énumération indépendante=5 couplages.


**Lecture du résultat.** Les deux machines indépendantes rendent le même compte : la
permanente de la matrice d'adjacence vaut exactement le nombre de couplages listés un par un.
Le pont est franchi — un comptage naturel est littéralement une permanente de matrice 0/1,
et par le théorème de Valiant, compter les couplages parfaits d'un graphe biparti est
une tâche #P-difficile dans le cas général : décider s'il en existe un est polynomial,
les compter ne l'est probablement pas.

Et l'histoire a continué : Aaronson et Arkhipov (2011) ont montré que la distribution de
sortie d'un réseau optique linéaire — des photons dans un interféromètre — dépend de
permanentes de matrices complexes, faisant de la fonction la plus dure du comptage classique
le candidat le plus sérieux d'un avantage quantique démontrable. Cette frontière est celle de
l'approfondissement [05b — La permanente, frontière quantique](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb).

### Exercice 3 — Couplages parfaits

Le graphe possède les arêtes $u_0-d_0$, $u_0-d_1$, $u_1-d_1$, $u_1-d_2$, $u_2-d_0$, $u_2-d_2$. Renseignez sa matrice 3×3 et prédisez son nombre de couplages. La cellule contrôle les arêtes et le compte.

In [9]:
int[,] matriceExercice=null; int? prediction=null; // TODO étudiant : matrice et nombre de couplages.
if(matriceExercice is null||prediction is null)Console.WriteLine("Exercice à compléter : fournir matrice et prédiction.");
else {var attendu=new int[,]{{1,1,0},{0,1,1},{1,0,1}};if(matriceExercice.GetLength(0)!=3||matriceExercice.GetLength(1)!=3)throw new Exception("Dimensions incorrectes");for(int i=0;i<3;i++)for(int j=0;j<3;j++)if(matriceExercice[i,j]!=attendu[i,j])throw new Exception("Une arête est incorrecte");long obtenu=Permanente(matriceExercice);if(prediction!=obtenu||obtenu!=2)throw new Exception("Prédiction incorrecte");Console.WriteLine($"Matrice du graphe vérifiée : {obtenu} couplages parfaits.");}

Exercice à compléter : fournir matrice et prédiction.


## Exercices

Les trois exercices du parcours portent sur la [définition](#exercice-1--compter-à-la-main), la [mesure](#exercice-2--léchelle-mesurée) et les [couplages](#exercice-3--couplages-parfaits). Ils restent à compléter.

## À retenir

| Question | Résultat |
|---|---|
| Vérifier ou compter ? | Quatre témoins sur huit certificats. |
| Déterminant | LU calcule le signe et le logarithme en temps cubique théorique. |
| Permanente | La définition exécute $n!$ termes ; la matrice 1 à 9 donne 450. |
| Difficulté | Valiant établit la #P-difficulté même en 0/1 ; cela ne démontre pas une borne factorielle universelle. |
| Couplages | Permanente et énumération indépendante donnent le même nombre. |

La suite [05b](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb) aborde la frontière quantique.

**Références** : Valiant (1979), *The Complexity of Computing the Permanent* ; Aaronson et Arkhipov (2011), *The Computational Complexity of Linear Optics* ; Minc (1978), *Permanents*.

## Vérifiez votre compréhension

1. Pourquoi le signe intervient-il dans le déterminant mais pas la permanente ?
2. Pourquoi ne peut-on pas déduire une borne factorielle universelle du banc ?
3. Quel lien unit couplages parfaits et permanente ?

<details><summary>Réponses</summary>

1. Le signe de la permutation produit les annulations du déterminant.
2. Le banc mesure une implémentation ; Valiant donne une difficulté de classe, pas une telle borne inconditionnelle.
3. Chaque permutation contribue 1 exactement quand elle définit un couplage parfait.

</details>

## Pour aller plus loin

- **Carnet suivant** : [Complexity-05b — La permanente, frontière quantique](Complexity-05b-AaronsonArkhipov-PermanenteBosonSampling-Python.ipynb) — la même fonction devient une loi de probabilité : $|\mathrm{perm}|^2$ gouverne la sortie d'un interféromètre, et le mur du comptage devient un candidat d'avantage quantique.
- **Dans la même série** : [Complexity-02 — Vérifier ou trouver](Complexity-02-P-NP-Reduction-Python.ipynb) — le mur exponentiel de l'énumération, côté existence ; [Complexity-06 — Simuler un circuit quantique classiquement](Complexity-06-Simuler-Circuit-Quantique-Classiquement-Python.ipynb) — l'autre bord de la frontière quantique.
- **Carnet voisin d'une autre série** : [ANALYSE-12 — Compter les couplages parfaits](../SymbolicAI/Lean/ANALYSE/ANALYSE-12-Comptage-Matching-Lean-Python.ipynb) — le même pont permanente ↔ couplages, cette fois avec une preuve formelle de l'autre côté.
- **Lectures** : les références complètes (Valiant 1979, Aaronson-Arkhipov 2011, Minc 1978) figurent en fin de section « À retenir ».
